# Fit Models

In [20]:
from bioacoustics.data import load_results

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score, hamming_loss

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Problems to think about:

- Multi-label prediction -> hierarchical models? implement with `Pipeline`
- Class imbalance -> tinker with class weights, adjust probabilities?
- Use additional metadata from train, not available in soundscapes
- Maybe use hybrid approaches? learn about how to combine different models' predictions

think about chunking the audio, tuning the threshold for ech class

## Load features and labels

In [21]:
# data = load_results("features", "data_train")
data = load_results("features", "data_train_soundscapes")

## Predict class

Start with predicting class only, we may after use this result when predicting primary label (hierarchical approach).



In [22]:
X = data["X"]
Y = data["y_class"]
# Y = data["y_primary"]

In [23]:
# clf = LogisticRegression(solver="liblinear", max_iter=1000, class_weight="balanced")
# clf = RandomForestClassifier(
#     n_estimators=200, max_depth=None, n_jobs=-1, class_weight="balanced"
# )
clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="logloss",
)

In [24]:
# TODO: try XGBoost, RF etc, change multi-label strategies
# TODO: hierarchical approach - train a model per class 
# and predict species conditioned on class (can try soft versions - multiply probas)

# TODO: alternative to hierarchical approach - include predicted class as a feature
# for species prediction
# TODO: the same idea for training metadata that is absent for the test
# - we can learn to predict this metadata as a secondary task and then include 
# it in the model
# NOTE: attention to data leak between train - validation when using secondary tasks 
# to generate features (class, metadata)

# TODO: can use this metadata as well for stratification - better split validation 
# (make sure that there is no data leak because of the same site)
# TODO: check whether metadata alone can predict species (check for shortcut learning)

# TODO: smart cross validations

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),  # otherwise MFCC and RMS are incomparable
        (
            "clf",
            OneVsRestClassifier(
                clf
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.","XGBClassifier...ree=None, ...)"
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'


In [25]:
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))
print(
    "Macro F1:", f1_score(y_test, y_pred, average="macro")
)  # F1 averaged across classes
print(
    "Micro F1:", f1_score(y_test, y_pred, average="micro")
)  # F1 on global TP, FP, FN, FP
print(
    "Hamming loss:", hamming_loss(y_test, y_pred)
)  # fraction of labels that are wrong

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        70
           1       0.89      0.86      0.88        37
           2       1.00      1.00      1.00        30
           3       0.71      0.83      0.77         6
           4       1.00      0.50      0.67         2

   micro avg       0.96      0.95      0.96       145
   macro avg       0.92      0.84      0.86       145
weighted avg       0.96      0.95      0.95       145
 samples avg       0.97      0.96      0.96       145

Macro F1: 0.8625219529329119
Micro F1: 0.9550173010380623
Hamming loss: 0.026


## Interpret the model

In [26]:
# TODO: get feature importance